In [1]:
import os
for r, d, f in os.walk('/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines'):
    print(r, '->', f)

/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines -> ['evaluate.py', '__results__.html', '__notebook__.ipynb', '__output__.json', 'results.csv', 'custom.css']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__pycache__ -> ['evaluate.cpython-312.pyc']
/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines/__results___files -> ['__results___5_0.png']


In [2]:
import sys
sys.path.insert(0, '/kaggle/input/notebooks/rayyanshuda/03-pytorch-and-baselines')
from evaluate import compute_metrics, evaluate_all

In [3]:
from torchvision import transforms as T

MEAN, STD, SIZE = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 160

train_tf = T.Compose([
    T.RandomResizedCrop(SIZE, scale=(0.6, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

eval_tf = T.Compose([
    T.Resize(SIZE),
    T.CenterCrop(SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

In [4]:
import os, torch, pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

print(os.listdir('/kaggle/input/notebooks/rayyanshuda/02-split-and-preprocess'))          # find your notebook-output folder name first

IN   = '/kaggle/input/notebooks/rayyanshuda/02-split-and-preprocess'
CSV  = f'{IN}/split_v1.csv'
IMGS = f'{IN}/resized'

class WildfireDataset(Dataset):
    def __init__(self, csv_path, img_dir, split, transform):
        # read the csv, keep only rows where split == this split
        df = pd.read_csv(csv_path)
        df = df[df.split == split].reset_index(drop=True)
        # TODO store what you need: the out_name list and the label list
        # label: fire -> 1.0, nofire -> 0.0   (float, for BCEWithLogitsLoss)
        self.files = df.out_name.tolist()
        self.labels = (df.cls == 'fire').astype('float32').tolist()
        # TODO store img_dir and transform
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        # how many examples in this split?
        return len(self.files) # DataLoader uses this to know the range of i

    def __getitem__(self, i):
        # open the image at img_dir/out_name[i], convert to RGB
        path = os.path.join(self.img_dir, self.files[i])
        img = Image.open(path).convert('RGB') # RGBA -> RGB, else 4-channel tensors
        # apply self.transform
        img = self.transform(img) # PIL -> augmented, normalized tensor
        label = torch.tensor(self.labels[i], dtype=torch.float32) # float, for BCEWithLogitsLoss
        return img, label

print(len(WildfireDataset(CSV, IMGS, 'train', train_tf)),
      len(WildfireDataset(CSV, IMGS, 'val',   eval_tf)),
      len(WildfireDataset(CSV, IMGS, 'test',  eval_tf)))    # expect 1803 420 476

['__results__.html', 'resized', 'inventory.csv', '__notebook__.ipynb', '__output__.json', 'split_v1.csv', 'custom.css']
1803 420 476


In [5]:
import torch
import torch.nn as nn

class ModelA(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.head(x)
        return x.squeeze(1)

m = ModelA()
xb, yb = next(iter(DataLoader(WildfireDataset(CSV, IMGS, 'train', train_tf), batch_size=32)))
print(m(xb).shape)                              # expect torch.Size([32])
print(sum(p.numel() for p in m.parameters()))   # expect 93601               # expect ~93601

torch.Size([32])
93601


In [6]:
import copy, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve, log_loss

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')


def predict(model, loader):
    """Return (y_true, y_score) as numpy. The only torch-aware piece."""
    model.eval()
    ys, ss = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            ss.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ss)


def pick_threshold(y_true, y_score):
    """Youden's J: maximise (sensitivity + specificity - 1). Chosen on VAL only."""
    fpr, tpr, thr = roc_curve(y_true, y_score)
    return float(thr[np.argmax(tpr - fpr)])


def train_model(make_model, seed, epochs=40, lr=1e-3, patience=8, bs=32,
                tag='model_a', limit=None):
    torch.manual_seed(seed); np.random.seed(seed)

    train_ds = WildfireDataset(CSV, IMGS, 'train', train_tf)
    val_ds   = WildfireDataset(CSV, IMGS, 'val',   eval_tf)
    if limit:                                   # dry-run mode
        g = torch.Generator().manual_seed(0)
        train_ds = Subset(train_ds, torch.randperm(len(train_ds), generator=g)[:limit].tolist())
        val_ds   = Subset(val_ds,   torch.randperm(len(val_ds),   generator=g)[:limit].tolist())

    train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2,
                          pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model     = make_model().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_auc, best_state, best_epoch, bad = -1, None, -1, 0
    history = []

    for epoch in range(epochs):
        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * len(yb); n += len(yb)
        scheduler.step()                      # once per EPOCH, not per batch

        yt, ys   = predict(model, val_dl)
        val_loss = log_loss(yt, np.clip(ys, 1e-7, 1 - 1e-7), labels=[0, 1])
        val_auc  = roc_auc_score(yt, ys)

        history.append({'epoch': epoch, 'train_loss': run_loss / n,
                        'val_loss': val_loss, 'val_auc': val_auc,
                        'lr': scheduler.get_last_lr()[0]})
        print(f'  ep {epoch:2d}  train {run_loss/n:.4f}  val {val_loss:.4f}  auc {val_auc:.4f}')

        if val_auc > best_auc:
            best_auc, best_epoch, bad = val_auc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())     # deepcopy, or it mutates
        else:
            bad += 1
            if bad >= patience:
                print(f'  early stop at {epoch}; best epoch was {best_epoch}')
                break

    model.load_state_dict(best_state)
    torch.save(best_state, f'/kaggle/working/{tag}_seed{seed}.pt')
    pd.DataFrame(history).to_csv(f'/kaggle/working/{tag}_seed{seed}_history.csv', index=False)

    yt_v, ys_v = predict(model, val_dl)
    return model, pd.DataFrame(history), best_auc, best_epoch, pick_threshold(yt_v, ys_v)

device: Tesla T4


In [7]:
test_dl   = DataLoader(WildfireDataset(CSV, IMGS, 'test', eval_tf),
                       batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_meta = pd.read_csv(CSV)
test_meta = test_meta[test_meta.split == 'test'].reset_index(drop=True)

EPOCHS = 80         # <-- set to 2 for a dry run, 40 for the real thing
all_rows = []

for seed in [0, 1, 2]:
    print(f'===== seed {seed} =====')
    model, hist, bauc, bep, thr = train_model(ModelA, seed, epochs=EPOCHS)
    print(f'  best val auc {bauc:.4f} @ epoch {bep} | threshold {thr:.3f}')
    yt, ys = predict(model, test_dl)
    rows = evaluate_all(yt, ys, test_meta.src.values, f'model_a_seed{seed}', threshold=thr)
    all_rows.append(rows)
    print(rows.to_string(index=False))

model_a_results = pd.concat(all_rows, ignore_index=True)
model_a_results.to_csv('/kaggle/working/model_a_results.csv', index=False)

print('\n===== mean ± spread across seeds =====')
print(model_a_results.groupby('slice')[['accuracy', 'roc_auc', 'recall', 'specificity']]
      .agg(['mean', 'std']).round(4).to_string())

===== seed 0 =====
  ep  0  train 0.5484  val 0.5498  auc 0.7819
  ep  1  train 0.5184  val 0.5150  auc 0.8128
  ep  2  train 0.5155  val 0.5250  auc 0.8058
  ep  3  train 0.4961  val 0.5144  auc 0.8153
  ep  4  train 0.4884  val 0.5192  auc 0.8221
  ep  5  train 0.4772  val 0.5209  auc 0.8148
  ep  6  train 0.4807  val 0.4942  auc 0.8377
  ep  7  train 0.4760  val 0.4779  auc 0.8481
  ep  8  train 0.4573  val 0.5510  auc 0.8308
  ep  9  train 0.4597  val 0.4763  auc 0.8437
  ep 10  train 0.4565  val 0.4929  auc 0.8448
  ep 11  train 0.4592  val 0.4944  auc 0.8313
  ep 12  train 0.4444  val 0.4682  auc 0.8533
  ep 13  train 0.4417  val 0.4775  auc 0.8462
  ep 14  train 0.4248  val 0.5556  auc 0.8364
  ep 15  train 0.4461  val 0.4782  auc 0.8472
  ep 16  train 0.4262  val 0.4862  auc 0.8563
  ep 17  train 0.4409  val 0.5561  auc 0.8450
  ep 18  train 0.4298  val 0.4632  auc 0.8606
  ep 19  train 0.4256  val 0.4489  auc 0.8666
  ep 20  train 0.4120  val 0.4466  auc 0.8740
  ep 21  train 